# 05 · GRU multihorizonte (Persona B)

**Fase CRISP-DM:** modelado y evaluación. Ventanas de **56 días pasados → 42 días futuros**, con una
segunda entrada de exógenas futuras conocidas (promos, festivos, `open_share`, calendario).

Reglas que se respetan: sin `Customers`; escalador ajustado solo con entrenamiento; las decisiones se
toman **con validación**; el conjunto de prueba se usa una sola vez.

> Pensado para **Google Colab / Kaggle con GPU** (Entorno de ejecución → Cambiar tipo → GPU).
> Toda la lógica está en `src/models/gru.py`; aquí solo se orquesta.

In [ ]:
# --- Preparación del entorno (Colab) ---
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("Rossman").exists():
        !git clone -b dev https://github.com/SantiagoTyrsArv/Rossman.git
    os.chdir("Rossman")
    !pip install -q pyarrow scikit-learn joblib
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.models import gru

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Datos
Se usa `artifacts/series/history.parquet` de Persona A. Si todavía no existe, se usan los datos de prueba de la app (solo para validar el flujo; **no** sirven para el informe).

In [ ]:
HIST = gru.ARTIFACTS / "series" / "history.parquet"
if HIST.exists():
    hist = pd.read_parquet(HIST)
    MOCK = False
else:
    from app import mock_data
    hist = mock_data.history()
    MOCK = True
    print("⚠️ Usando DATOS DE PRUEBA: no escribir artifacts finales con esto.")
hist["ds"] = pd.to_datetime(hist["ds"])
hist.groupby("unique_id")["ds"].agg(["min", "max", "count"])

## 2. v1: serie `total`

In [ ]:
UID = "total"
sd = gru.prepare_series(hist[hist["unique_id"] == UID], fit_end=gru.TRAIN_END)
(xp_tr, xf_tr, y_tr), (xp_va, xf_va, y_va) = gru.train_val_windows(sd)
print("train:", xp_tr.shape, xf_tr.shape, y_tr.shape, "| val:", xp_va.shape)

In [ ]:
def fit(sd, units=64, lookback=56, dropout=0.2, epochs=300, batch_size=32, verbose=0):
    (xp_tr, xf_tr, y_tr), (xp_va, xf_va, y_va) = gru.train_val_windows(sd, lookback=lookback)
    model = gru.build_model(lookback=lookback, units=units, dropout=dropout)
    h = model.fit([xp_tr, xf_tr], y_tr, validation_data=([xp_va, xf_va], y_va),
                  epochs=epochs, batch_size=batch_size, callbacks=gru.default_callbacks(),
                  verbose=verbose, shuffle=True)
    return model, h


def predict_origin(model, sd, origin, lookback=56, fut=None, mc=100):
    fut = sd.fut if fut is None else fut
    xp, xf = gru.origin_input(sd.past, fut, sd.idx(origin), lookback=lookback)
    mean, lo, hi = gru.predict_mc(model, [xp, xf], n_samples=mc)
    inv = lambda a: gru.inverse_target(a[0], sd.scaler)
    return inv(mean), inv(lo), inv(hi)


def val_metrics(model, sd, lookback=56):
    y_hat, _, _ = predict_origin(model, sd, gru.TRAIN_END, lookback, mc=30)
    i = sd.idx(gru.TRAIN_END)
    return gru.compute_all(sd.y[i + 1 : i + 43], y_hat)


model, h = fit(sd, verbose=1)
plt.figure(figsize=(8, 3))
plt.plot(h.history["loss"], label="train"); plt.plot(h.history["val_loss"], label="val")
plt.yscale("log"); plt.legend(); plt.title("Curvas de aprendizaje (Huber)"); plt.show()
best_epoch = int(np.argmin(h.history["val_loss"])) + 1
print("mejor época:", best_epoch, "| validación:", val_metrics(model, sd))

**Comentario de las curvas (completar):** ¿la pérdida de validación se separa de la de entrenamiento
(sobreajuste)? Con ~900 días hay pocas ventanas: si v1 sobreajusta o rinde menos que SARIMAX, pasar a
**v2 (modelo global)**, registrándolo antes como propuesta en `docs/PROPUESTAS.md`.

## 3. Variantes (se elige por validación)

In [ ]:
GRID = [
    {"units": 32, "lookback": 56, "dropout": 0.2},
    {"units": 64, "lookback": 56, "dropout": 0.2},
    {"units": 128, "lookback": 56, "dropout": 0.3},
    {"units": 64, "lookback": 28, "dropout": 0.2},
]
results = []
for params in GRID:
    m, hh = fit(sd, **params)
    met = val_metrics(m, sd, params["lookback"])
    ep = int(np.argmin(hh.history["val_loss"])) + 1
    results.append({**params, **met, "best_epoch": ep})
    if not MOCK:
        gru.log_experiment({**params, "uid": UID, "best_epoch": ep}, met)
res = pd.DataFrame(results).sort_values("mape")
best = res.iloc[0].to_dict()
res

## 4. Predicciones con origen fijo (val, test, future)

In [ ]:
P = {k: int(best[k]) if k != "dropout" else float(best[k]) for k in ("units", "lookback", "dropout")}
EPOCHS = int(best["best_epoch"])
rows = []

# val: entrenado con objetivos <= TRAIN_END (EarlyStopping con la validación)
m_val, _ = fit(sd, **P)
y_hat, lo, hi = predict_origin(m_val, sd, gru.TRAIN_END, P["lookback"])
d = gru.split_dates(gru.TRAIN_END)
rows.append(gru.forecast_frame(UID, d, sd.y[sd.idx(d[0]) : sd.idx(d[-1]) + 1], y_hat, "val", lo, hi))


def refit_until(sd, end, epochs):
    # Reentrena con todas las ventanas cuyo objetivo termina <= end, con el número de épocas
    # elegido en validación (no hay validación aquí: el escalador sigue siendo el de TRAIN_END).
    xp, xf, yy = gru.make_windows(sd.past, sd.fut, sd.y_scaled, P["lookback"], 42, sd.idx(end))
    m = gru.build_model(lookback=P["lookback"], units=P["units"], dropout=P["dropout"])
    m.fit([xp, xf], yy, epochs=epochs, batch_size=32, verbose=0)
    return m


# test: datos hasta VAL_END
m_test = refit_until(sd, gru.VAL_END, EPOCHS)
y_hat, lo, hi = predict_origin(m_test, sd, gru.VAL_END, P["lookback"])
d = gru.split_dates(gru.VAL_END)
rows.append(gru.forecast_frame(UID, d, sd.y[sd.idx(d[0]) : sd.idx(d[-1]) + 1], y_hat, "test", lo, hi))

In [ ]:
# future: datos hasta TEST_END + exógenas futuras de build_future_exog (Persona A)
d = gru.split_dates(gru.TEST_END)
try:
    fut_exog = gru.future_exog_matrix(UID, d)
    fut_all = np.vstack([sd.fut, fut_exog])
    m_fut = refit_until(sd, gru.TEST_END, EPOCHS)
    y_hat, lo, hi = predict_origin(m_fut, sd, gru.TEST_END, P["lookback"], fut=fut_all)
    rows.append(gru.forecast_frame(UID, d, None, y_hat, "future", lo, hi))
except ImportError:
    print("build_future_exog de A todavía no existe: se omite el split future.")

fc = pd.concat(rows, ignore_index=True)
fc.groupby("split")[["y", "y_hat"]].sum()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
recent = hist[(hist.unique_id == UID) & (hist.ds > gru.TRAIN_END - pd.Timedelta(days=60))]
ax.plot(recent.ds, recent.y, color="k", lw=1, label="real")
for split, g in fc.groupby("split"):
    ax.plot(g.ds, g.y_hat, lw=2, label=f"GRU {split}")
    ax.fill_between(g.ds, g.lo_95, g.hi_95, alpha=0.15)
ax.legend(); ax.set_title("GRU · total"); plt.show()

## 5. Escribir artifacts (solo con datos reales)

In [ ]:
if MOCK:
    print("Datos de prueba: no se escriben artifacts.")
else:
    metrics = gru.write_artifacts(fc)
    gru.save_scaler(sd.scaler)
    print(metrics)

In [ ]:
# Modelo .keras: fuera de artifacts/ (Drive). Solo si pesa < 25 MB se puede guardar en artifacts/models/.
if IN_COLAB and not MOCK:
    from google.colab import drive
    drive.mount("/content/drive")
    m_fut.save("/content/drive/MyDrive/rossmann_gru.keras")

> Después de escribir los artifacts en Colab: descargar `artifacts/forecasts/gru.parquet`,
> `artifacts/metrics/gru.json` y `artifacts/models/gru_scaler.joblib`, copiarlos al repo local y
> hacer commit en una rama `feature/b-gru` (o hacer `git push` desde Colab con un token).